# 06 · Are the results real? Leakage, memorisation and significance checks

The foundation models score well, sometimes better than XGBoost on the same rows. Before believing that, this notebook rules out four ways a score could be fake:

| Check | Question | Passes if |
| --- | --- | --- |
| 1. Index leakage | Did any validation or test row get into a context? | every count is 0 |
| 2. Duplicate rows | ULB has 1,081 duplicated transactions. Does recognising an exact copy explain the scores? | PR-AUC barely changes when test rows with a copy in training are removed |
| 3. Shuffled labels | Does the model really use the labels it is shown, or does it "know" ULB from pretraining? | with shuffled labels PR-AUC falls to the fraud rate |
| 4. Significance | Is the TFM vs XGBoost (same rows) difference bigger than chance? | pooled 95% interval excludes 0 after Holm correction |

Run after notebooks 03 and 05. Checks 1, 2 and 4 only read saved scores. Check 3 fits a few models, so use a **T4 GPU** (about 15 minutes).

In [ ]:
# 1. Settings.
SHUFFLE_LEVELS = ["natural", "1"]        # fraud rates for the shuffled-label check
SHUFFLE_K = 10_000                         # context size for that check (seed 0's saved uniform context)
SHUFFLE_MODELS = ["TabICLv2", "TabPFN v2"]

In [ ]:
# 2. Drive, latest code, libraries.
from google.colab import drive
drive.mount("/content/drive")
import os, sys
REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
# 3. Tests, data, and the split fingerprint check.
!cd {REPO_DIR} && PYTHONPATH=src python -m pytest tests -q
from fraudicl.config import DRIVE_DIR
from fraudicl.data import load_ulb, features_and_target, load_splits, fingerprint
X, y = features_and_target(load_ulb())
splits = load_splits(f"{DRIVE_DIR}/splits/ulb_splits.npz")
expected = open(f"{REPO_DIR}/results/ulb_splits_fingerprint.txt").readline().strip()
assert fingerprint(splits) == expected, "splits on Drive differ from the ones on record"
OUT = f"{DRIVE_DIR}/results"
import pandas as pd
from fraudicl import checks as C
from fraudicl.report import load_runs
print("splits OK", expected)

In [ ]:
# 4. Check 1: no context ever contains a validation or test row.
leak = C.context_leakage(OUT, splits)
print(len(leak), "saved contexts checked")
print(leak[["outside_train", "in_validation", "in_test"]].sum().to_string())
assert (leak[["outside_train", "in_validation", "in_test"]] == 0).all().all(), "LEAK FOUND"
print("check 1 passed: every context is made of training rows only")

In [ ]:
# 5. Check 2a: how many test rows have an exact copy in the training rows?
dups = C.duplicate_overlap(X, y, splits)
dup_summary = (dups.groupby("level")[["test_rows", "test_rows_with_copy_in_train",
                                      "test_frauds", "test_frauds_with_copy_in_train"]]
                   .mean().round(1).reindex(C.LEVEL_ORDER))
dup_summary

In [ ]:
# 6. Check 2b: PR-AUC with those test rows removed (same saved scores, nothing refitted).
#    Uniform-context TFMs, matched XGBoost and full-data XGBoost (plain).
res = C.rescore_without_duplicates(OUT, X, y, splits)
res = res[(res.policy.isin(["uniform", "all rows"])) | (res.family == "xgb")]
res = res[~((res.family == "xgb") & (~res.model.eq("XGBoost (plain)")))].copy()
res.loc[res.family == "xgb", "k"] = float("nan")   # full-data XGBoost: k is its training size, not a context
dedup = (res.groupby(["model", "level", "k"], dropna=False)
            .agg(n_seeds=("seed", "nunique"), frauds_dropped=("frauds_dropped", "mean"),
                 pr_auc=("pr_auc", "mean"), pr_auc_without_duplicates=("pr_auc_without_duplicates", "mean"),
                 change=("change", "mean"))
            .reset_index())
dedup["level"] = pd.Categorical(dedup.level, C.LEVEL_ORDER, ordered=True)
dedup = dedup.sort_values(["model", "level", "k"]).round(4)
print("largest change in mean PR-AUC:", dedup.change.abs().max())
dedup

**How to read check 2.** `frauds_dropped` is how many of the ~98 test frauds had an exact copy in training. If `change` is small for every model (well under the seed std of 0.01-0.05), duplicates don't explain any result. If a TFM's PR-AUC dropped much more than XGBoost's, the TFM was partly recognising copies, and the paper must report the de-duplicated numbers instead.

In [ ]:
# 7. Check 3: shuffled labels (GPU). Same context rows as the seed-0 uniform run, labels permuted.
from fraudicl.models import make_tabpfn_v2, make_tabicl_v2
from fraudicl.experiments import get_context
makers = {"TabICLv2": make_tabicl_v2, "TabPFN v2": make_tabpfn_v2}
by_key = {(s.seed, s.level): s for s in splits}
shuffle = []
for level in SHUFFLE_LEVELS:
    s = by_key[0, level]
    ctx = get_context(OUT, s, y, SHUFFLE_K, "uniform", 0)
    for m in SHUFFLE_MODELS:
        print(m, level, flush=True)
        shuffle.append(C.shuffled_label_control(X, y, s, ctx, makers[m], m))
shuffle = pd.concat(shuffle).round(4)
shuffle.to_csv(f"{OUT}/check_shuffled_labels.csv", index=False)
shuffle

**How to read check 3.** With real labels you should see the usual scores (about 0.76 at 0.172%, 0.87 at 1%). With shuffled labels the model has the same rows but the fraud labels sit on random rows, so a model that learns from its context can only guess: PR-AUC near `no_skill_pr_auc` (the fraud rate). If it stays high, the model would be using knowledge from outside the context, i.e. memorisation.

In [ ]:
# 8. Check 4: TFM vs XGBoost trained on the same k rows, every fraud rate and k (CPU, ~5 minutes).
runs = load_runs(OUT)
sig = C.matched_tests(OUT, y, runs)
sig.to_csv(f"{OUT}/check_tfm_vs_matched_xgb.csv", index=False)
sig.round(4)

**How to read check 4.** One row per model, fraud rate and k.

- `mean_diff`: TFM minus XGBoost on the same rows, averaged over the 5 seeds. Positive = TFM better.
- `seeds_tfm_ahead`: out of 5.
- `ci_low`, `ci_high`: 95% interval of the mean difference, pooling the test sets of all seeds. If it excludes 0 the difference is bigger than test-set noise.
- `p_value`, `p_holm`: the second is corrected for testing ~32 cells at once. Only `p_holm < 0.05` counts as significant. A `p_value` of 0 means below 1/1000 (no bootstrap round crossed zero).

This interval covers which test rows were drawn, not which contexts were drawn. A claim that holds here is still exploratory, because we spotted the small-k pattern after looking at these results. The confirmatory run with fresh contexts comes next.

In [ ]:
# 9. Print everything to paste back.
print("CHECK 1 (leak counts, must be 0)"); print(leak[["outside_train", "in_validation", "in_test"]].sum().to_csv())
print("CHECK 2a"); print(dup_summary.to_csv())
print("CHECK 2b"); print(dedup.to_csv(index=False))
print("CHECK 3"); print(shuffle.to_csv(index=False))
print("CHECK 4"); print(sig.round(4).to_csv(index=False))